In [1]:
# Setup imports
%matplotlib inline
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import Markdown as md
import time
import os
import serial
from pico import Pico

import serial.tools.list_ports
for port in serial.tools.list_ports.comports():
    print(port.device, port.description) # this should show the port where the Pi Pico

# sound feedback. use by calling beep(<number of beeps>)
beep = lambda x: os.system("echo -n '\a';sleep 0.2;" * x)

/dev/ttyACM0 Board in FS mode - Board CDC


In [2]:
%%javascript
IPython.OutputArea.prototype._should_scroll = function(lines) {
    return false;
}

<IPython.core.display.Javascript object>

In [3]:
# Make graphs large enough to read easily
plt.clf()
plt.rcParams["figure.figsize"] = [10.0, 10.0*2/3]
matplotlib.rcParams["font.size"] = 20
None

# loads caen_daq code
display(md("### load the code"))
%run gui/wja_caen_tcal
tbegin = time.time()

# loads the drs4 corrections
f.load_drs_corrections()
print(list(f.tcal[0].__dict__.keys()))

### load the code

wja_caen_tcal.py starting at Fri Aug 28 15:11:53 2026


<Figure size 640x480 with 0 Axes>

['idrs', 'ichnl', 'tstamp', 'cellpeds', 'cellgain', 'celldt', 'wiggleshape', 'meanshape']


<Figure size 1000x666.667 with 0 Axes>

In [4]:
pico = Pico("/dev/ttyACM0")   # replace with the actual port of the Pi Pico
print(repr(pico.exec("import Main as m"))) # loads Main.py from the Pico
pico.exec("import sys; del sys.modules['Main']; import Main as m") # this is a trick to reload the module in case it was already loaded, so that we get the latest version after uploading
print(repr(pico.exec("print(m.current_M_yaxis)"))) # Prints current M from y axis, just to test that things worked
print(pico.exec("m.enable_motor()")) # engages the motors. they should be hard to turn manually after this

✓ Uploaded Main.py → Main.py (23261 bytes)
<module 'Main' from 'Main.py'>
'import Main as m\r\n>>> '
0.0
'print(m.current_M_yaxis)\r\n0.0\r\n>>> '
m.enable_motor()
m.enable_motor()
>>> 


In [5]:
# REUPLOAD AND RELOAD
pico.upload("/home/emmo/proj/sipm/PiPico_MotorControls/Main.py")
pico.exec("import sys; del sys.modules['Main']; import Main as m")

✓ Uploaded /home/emmo/proj/sipm/PiPico_MotorControls/Main.py → Main.py (23261 bytes)


"import sys; del sys.modules['Main']; import Main as m\r\n>>> "

In [ ]:
pico.exec("m.calibrate('x')") # calibrate the x axis to setup its range of motion

In [ ]:
pico.exec("m.calibrate('y')") # calibrate the y axis to setup its range of motion

In [6]:
# X coordinates from home
# pico.exec("m.move_distance_mm_x(20.55, True, True)") # col 1
x0 = 20.55
# pico.exec("m.move_distance_mm_x(19.05, True, True)") # col 2
x1 = 19.05
# pico.exec("m.move_distance_mm_x(17.47, True, True)") # col 3
x2 = 17.47
# pico.exec("m.move_distance_mm_x(15.97, True, True)") # col 4
x3 = 15.97
# pico.exec("m.move_distance_mm_x(14.39, True, True)") # col 5
x4 = 14.39
# pico.exec("m.move_distance_mm_x(12.89, True, True)") # col 6
x5 = 12.89
# pico.exec("m.move_distance_mm_x(11.31, True, True)") # col 7
x6 = 11.31
# pico.exec("m.move_distance_mm_x(9.81, True, True)") # col 8
x7 = 9.81
# pico.exec("m.move_distance_mm_x(8.23, True, True)") # col 9
x8 = 8.23
# pico.exec("m.move_distance_mm_x(6.73, True, True)") # col 10
x9 = 6.73
# pico.exec("m.move_distance_mm_x(13.61, True, True)") # xtal array center x
x_center = 13.61

# Y coordinates from home
# pico.exec("m.move_distance_mm_y(44.65, True, True)") # row 1
y0 = 44.65
# pico.exec("m.move_distance_mm_y(43.15, True, True)") # row 2
y1 = 43.15
# pico.exec("m.move_distance_mm_y(41.57, True, True)") # row 3
y2 = 41.57
# pico.exec("m.move_distance_mm_y(40.07, True, True)") # row 4
y3 = 40.07
# pico.exec("m.move_distance_mm_y(38.49, True, True)") # row 5
y4 = 38.49
# pico.exec("m.move_distance_mm_y(36.99, True, True)") # row 6
y5 = 36.99
# pico.exec("m.move_distance_mm_y(35.41, True, True)") # row 7
y6 = 35.41
# pico.exec("m.move_distance_mm_y(33.91, True, True)") # row 8
y7 = 33.91
# pico.exec("m.move_distance_mm_y(32.33, True, True)") # row 9
y8 = 32.33
# pico.exec("m.move_distance_mm_y(30.83, True, True)") # row 10
y9 = 30.83
# pico.exec("m.move_distance_mm_y(37.74, True, True)") # xtal arraycenter y
y_center = 37.74



In [ ]:
# moving to xtal 100 center
pico.exec(f"m.move_distance_mm_x({x9}, True, True)") # (distance in mm, True for fast speed, True for forward)
pico.exec(f"m.move_distance_mm_y({y9}, True, True)") # (distance in mm, True for fast speed, True for forward)
# x: 6.73mm 
# y: 30.83mm

In [ ]:
# acquire
starttime = time.time()
nevents = 1000
t0 = time.time()
f.do_triggered_readout(nevents)
print(list(f.trigev[0].__dict__.keys()))
print(f"acquisition elapsed time {time.time()-t0:.1f}s")
print(f"rough rate = {nevents/(time.time()-t0):.1f}Hz")
beep(1)
t0 = time.time()
f.correct_triggered_readout()
print(f"corrections elapsed time {time.time()-t0:.1f}s")
beep(1)

# plot 
for ich in [0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16]:
    print(f"SiPM Channel {ich+1} of 16")
    for ev in range (min(1000, nevents)):
        plt.plot(f.trigev[ev].drsu[ich])
    plt.show()
    print(f"----------------------")
    
t0 = time.time()

# Save to HDF5 file
hdf5_fnam = f"20260511_hfv2_TAC1p9x19_44p5V_Ref100mV_Sum000mV_100e_pos{position}_CAEN.hdf5"

ns = SimpleNamespace()
ns.drsraw = np.array([e.drsraw for e in f.trigev], dtype=np.int16)
ns.drs = np.array([e.drsgc for e in f.trigev], dtype=np.float32)
ns.drsu = np.array([e.drsu for e in f.trigev], dtype=np.float32)
ns.traw = np.array([e.traw for e in f.trigev], dtype=np.float32)
ns.tcor = np.array([e.tcor for e in f.trigev], dtype=np.float32)
ns.drs_trig_cell = np.array([e.drs_trig_cell for e in f.trigev], dtype=np.int16)
ns.drs_tstamp = np.array([e.tstamp for e in f.trigev], dtype=np.int64)
ns.drs_cellgains = np.array([tc.cellgain for tc in f.tcal], dtype=np.float32)
ns.drs_cellwidth = np.array([tc.celldt for tc in f.tcal], dtype=np.float32)
ns.drs_peds = np.array([tc.cellpeds for tc in f.tcal], dtype=np.float32)
ns.drs_wiggle_shape = np.array([tc.wiggleshape for tc in f.tcal], dtype=np.float32)
ns.drs_mean_shape = np.array([tc.meanshape for tc in f.tcal], dtype=np.float32)

if "hf" in vars():
    hf.close()
    del hf

hf = h5py.File(hdf5_fnam, "w")

def cds(dsetname, comment):
    ds = hf.create_dataset(
        dsetname, 
        data=ns.__dict__[dsetname],
        shuffle=True,
        compression="gzip",
        compression_opts=1)
    ds.attrs["comment"] = comment

cds("drsraw", "raw uncorrected DRS samples [event][channel][sample]")
cds("drs", "DRS samples with pedestal and gain corrections applied")
cds("drsu", "DRS corrected samples, resampled to equal time intervals")
cds("traw", "nominal uncorrected DRS sample times")
cds("tcor", "DRS sample times, corrected via timing calibration")
cds("drs_trig_cell", "DRS trigger/stop cell ID [event][channel]")
cds("drs_tstamp", "CAEN board time stamp [event]")
cds("drs_cellgains", "voltage gain [channel][cell] from DRS calibration")
cds("drs_cellwidth", "cell width (ns) [channel][cell] from DRS timing calibration")
cds("drs_peds", "DRS pedestal [channel][cell] from DRS calibration")
cds("drs_wiggle_shape", "fitted 'wiggle' shape subtracted from each DRS waveform")
cds("drs_mean_shape", "mean waveform, in absence of signal, CAEN board artifact")
hf.close()
del hf
print(f"hdf5 elapsed time: {time.time()-t0:.1f}s")
print(f"total elapsed time {time.time()-starttime:.1f}s")

In [ ]:

starttime = time.time()
nevents = 1000
t0 = time.time()
f.do_triggered_readout(nevents)
print(list(f.trigev[0].__dict__.keys()))
print(f"acquisition elapsed time {time.time()-t0:.1f}s")
print(f"rough rate = {nevents/(time.time()-t0):.1f}Hz")
beep(1)
t0 = time.time()
f.correct_triggered_readout()
print(f"corrections elapsed time {time.time()-t0:.1f}s")
beep(1)
for ich in [0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15]:
# for ich in [0,1,2,4,5,6,13,14,15]:
# for ich in [16]:
    print(f"CAEN Channel {ich+1} of 16")
    for ev in range (nevents):
        plt.plot(f.trigev[ev].drsu[ich])
    plt.show()
    print(f"----------------------")
print(f"total elapsed time {time.time()-starttime:.1f}s")


In [ ]:
starttime = time.time()
nevents = 100
t0 = time.time()
f.do_triggered_readout(nevents)
print(list(f.trigev[0].__dict__.keys()))
print(f"acquisition elapsed time {time.time()-t0:.1f}s")
print(f"rough rate = {nevents/(time.time()-t0):.1f}Hz")
beep(1)
t0 = time.time()
f.correct_triggered_readout()
print(f"corrections elapsed time {time.time()-t0:.1f}s")
beep(1)
# for ich in [0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15]:
for ich in [0,1,2,4,5,6,13,14,15]:
# for ich in [16]:
    print(f"CAEN Channel {ich+1} of 16")
    for ev in range (nevents):
        plt.plot(f.trigev[ev].drsu[ich])
    plt.show()
    print(f"----------------------")
print(f"total elapsed time {time.time()-starttime:.1f}s")


In [ ]:
starttime = time.time()
nevents = 200
t0 = time.time()
f.do_triggered_readout(nevents)
print(list(f.trigev[0].__dict__.keys()))
print(f"acquisition elapsed time {time.time()-t0:.1f}s")
print(f"rough rate = {nevents/(time.time()-t0):.1f}Hz")
beep(1)
t0 = time.time()
f.correct_triggered_readout()
print(f"corrections elapsed time {time.time()-t0:.1f}s")
beep(1)
for ich in [0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15]:
# for ich in [0,1,2,4,5,6,13,14,15]:
# for ich in [16]:
    print(f"CAEN Channel {ich+1} of 16")
    for ev in range (nevents):
        plt.plot(f.trigev[ev].drsu[ich])
    plt.show()
    print(f"----------------------")
print(f"total elapsed time {time.time()-starttime:.1f}s")


In [ ]:
beep(2)
time.sleep(50) # oops

In [ ]:
# Setup variables for all 16 positions
# nevents = 1000
# position = 0

In [ ]:
# acquire and plot
starttime = time.time()
nevents = 1000
t0 = time.time()
f.do_triggered_readout(nevents)
print(list(f.trigev[0].__dict__.keys()))
print(f"acquisition elapsed time {time.time()-t0:.1f}s")
print(f"rough rate = {nevents/(time.time()-t0):.1f}Hz")
beep(1)
t0 = time.time()
f.correct_triggered_readout()
print(f"corrections elapsed time {time.time()-t0:.1f}s")
beep(1)

for ich in [0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16]:
    print(f"SiPM Channel {ich+1} of 16")
    for ev in range (min(1000, nevents)):
        plt.plot(f.trigev[ev].drsu[ich])
    plt.show()
    print(f"----------------------")
    
t0 = time.time()
hdf5_fnam = f"20260511_hfv2_TAC1p9x19_44p5V_Ref100mV_Sum000mV_100e_pos{position}_CAEN.hdf5"

ns = SimpleNamespace()
ns.drsraw = np.array([e.drsraw for e in f.trigev], dtype=np.int16)
ns.drs = np.array([e.drsgc for e in f.trigev], dtype=np.float32)
ns.drsu = np.array([e.drsu for e in f.trigev], dtype=np.float32)
ns.traw = np.array([e.traw for e in f.trigev], dtype=np.float32)
ns.tcor = np.array([e.tcor for e in f.trigev], dtype=np.float32)
ns.drs_trig_cell = np.array([e.drs_trig_cell for e in f.trigev], dtype=np.int16)
ns.drs_tstamp = np.array([e.tstamp for e in f.trigev], dtype=np.int64)
ns.drs_cellgains = np.array([tc.cellgain for tc in f.tcal], dtype=np.float32)
ns.drs_cellwidth = np.array([tc.celldt for tc in f.tcal], dtype=np.float32)
ns.drs_peds = np.array([tc.cellpeds for tc in f.tcal], dtype=np.float32)
ns.drs_wiggle_shape = np.array([tc.wiggleshape for tc in f.tcal], dtype=np.float32)
ns.drs_mean_shape = np.array([tc.meanshape for tc in f.tcal], dtype=np.float32)

if "hf" in vars():
    hf.close()
    del hf

hf = h5py.File(hdf5_fnam, "w")

def cds(dsetname, comment):
    ds = hf.create_dataset(
        dsetname, 
        data=ns.__dict__[dsetname],
        shuffle=True,
        compression="gzip",
        compression_opts=1)
    ds.attrs["comment"] = comment

cds("drsraw", "raw uncorrected DRS samples [event][channel][sample]")
cds("drs", "DRS samples with pedestal and gain corrections applied")
cds("drsu", "DRS corrected samples, resampled to equal time intervals")
cds("traw", "nominal uncorrected DRS sample times")
cds("tcor", "DRS sample times, corrected via timing calibration")
cds("drs_trig_cell", "DRS trigger/stop cell ID [event][channel]")
cds("drs_tstamp", "CAEN board time stamp [event]")
cds("drs_cellgains", "voltage gain [channel][cell] from DRS calibration")
cds("drs_cellwidth", "cell width (ns) [channel][cell] from DRS timing calibration")
cds("drs_peds", "DRS pedestal [channel][cell] from DRS calibration")
cds("drs_wiggle_shape", "fitted 'wiggle' shape subtracted from each DRS waveform")
cds("drs_mean_shape", "mean waveform, in absence of signal, CAEN board artifact")
hf.close()
del hf
print(f"hdf5 elapsed time: {time.time()-t0:.1f}s")
print(f"total elapsed time {time.time()-starttime:.1f}s")

In [ ]:
# move to position 2
position = "02"
pico.exec("m.move_distance_mm_y(4, speed_fast=True, forward=True)")
pico.exec("m.move_distance_mm_x(4, speed_fast=True, forward=True)")

# acquire and plot
starttime = time.time()
nevents = 2000
t0 = time.time()
f.do_triggered_readout(nevents)
print(list(f.trigev[0].__dict__.keys()))
print(f"acquisition elapsed time {time.time()-t0:.1f}s")
print(f"rough rate = {nevents/(time.time()-t0):.1f}Hz")
beep(1)
t0 = time.time()
f.correct_triggered_readout()
print(f"corrections elapsed time {time.time()-t0:.1f}s")
beep(1)

for ich in [0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16]:
    print(f"SiPM Channel {ich+1} of 16")
    for ev in range (min(1000, nevents)):
        plt.plot(f.trigev[ev].drsu[ich])
    plt.show()
    print(f"----------------------")
    
t0 = time.time()
hdf5_fnam = f"20260511_hfv2_TAC1p9x19_44p5V_Ref100mV_Sum000mV_100e_pos{position}_CAEN.hdf5"

ns = SimpleNamespace()
ns.drsraw = np.array([e.drsraw for e in f.trigev], dtype=np.int16)
ns.drs = np.array([e.drsgc for e in f.trigev], dtype=np.float32)
ns.drsu = np.array([e.drsu for e in f.trigev], dtype=np.float32)
ns.traw = np.array([e.traw for e in f.trigev], dtype=np.float32)
ns.tcor = np.array([e.tcor for e in f.trigev], dtype=np.float32)
ns.drs_trig_cell = np.array([e.drs_trig_cell for e in f.trigev], dtype=np.int16)
ns.drs_tstamp = np.array([e.tstamp for e in f.trigev], dtype=np.int64)
ns.drs_cellgains = np.array([tc.cellgain for tc in f.tcal], dtype=np.float32)
ns.drs_cellwidth = np.array([tc.celldt for tc in f.tcal], dtype=np.float32)
ns.drs_peds = np.array([tc.cellpeds for tc in f.tcal], dtype=np.float32)
ns.drs_wiggle_shape = np.array([tc.wiggleshape for tc in f.tcal], dtype=np.float32)
ns.drs_mean_shape = np.array([tc.meanshape for tc in f.tcal], dtype=np.float32)

if "hf" in vars():
    hf.close()
    del hf

hf = h5py.File(hdf5_fnam, "w")

def cds(dsetname, comment):
    ds = hf.create_dataset(
        dsetname, 
        data=ns.__dict__[dsetname],
        shuffle=True,
        compression="gzip",
        compression_opts=1)
    ds.attrs["comment"] = comment

cds("drsraw", "raw uncorrected DRS samples [event][channel][sample]")
cds("drs", "DRS samples with pedestal and gain corrections applied")
cds("drsu", "DRS corrected samples, resampled to equal time intervals")
cds("traw", "nominal uncorrected DRS sample times")
cds("tcor", "DRS sample times, corrected via timing calibration")
cds("drs_trig_cell", "DRS trigger/stop cell ID [event][channel]")
cds("drs_tstamp", "CAEN board time stamp [event]")
cds("drs_cellgains", "voltage gain [channel][cell] from DRS calibration")
cds("drs_cellwidth", "cell width (ns) [channel][cell] from DRS timing calibration")
cds("drs_peds", "DRS pedestal [channel][cell] from DRS calibration")
cds("drs_wiggle_shape", "fitted 'wiggle' shape subtracted from each DRS waveform")
cds("drs_mean_shape", "mean waveform, in absence of signal, CAEN board artifact")
hf.close()
del hf
print(f"hdf5 elapsed time: {time.time()-t0:.1f}s")
print(f"total elapsed time {time.time()-starttime:.1f}s")

In [ ]:
# move to position 3
position = "03"
pico.exec("m.move_distance_mm_y(4, speed_fast=True, forward=True)")
pico.exec("m.move_distance_mm_x(4, speed_fast=True, forward=True)")

# acquire and plot
starttime = time.time()
nevents = 3000
t0 = time.time()
f.do_triggered_readout(nevents)
print(list(f.trigev[0].__dict__.keys()))
print(f"acquisition elapsed time {time.time()-t0:.1f}s")
print(f"rough rate = {nevents/(time.time()-t0):.1f}Hz")
beep(1)
t0 = time.time()
f.correct_triggered_readout()
print(f"corrections elapsed time {time.time()-t0:.1f}s")
beep(1)

for ich in [0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16]:
    print(f"SiPM Channel {ich+1} of 16")
    for ev in range (min(1000, nevents)):
        plt.plot(f.trigev[ev].drsu[ich])
    plt.show()
    print(f"----------------------")
    
t0 = time.time()
hdf5_fnam = f"20260511_hfv2_TAC1p9x19_44p5V_Ref100mV_Sum000mV_100e_pos{position}_CAEN.hdf5"

ns = SimpleNamespace()
ns.drsraw = np.array([e.drsraw for e in f.trigev], dtype=np.int16)
ns.drs = np.array([e.drsgc for e in f.trigev], dtype=np.float32)
ns.drsu = np.array([e.drsu for e in f.trigev], dtype=np.float32)
ns.traw = np.array([e.traw for e in f.trigev], dtype=np.float32)
ns.tcor = np.array([e.tcor for e in f.trigev], dtype=np.float32)
ns.drs_trig_cell = np.array([e.drs_trig_cell for e in f.trigev], dtype=np.int16)
ns.drs_tstamp = np.array([e.tstamp for e in f.trigev], dtype=np.int64)
ns.drs_cellgains = np.array([tc.cellgain for tc in f.tcal], dtype=np.float32)
ns.drs_cellwidth = np.array([tc.celldt for tc in f.tcal], dtype=np.float32)
ns.drs_peds = np.array([tc.cellpeds for tc in f.tcal], dtype=np.float32)
ns.drs_wiggle_shape = np.array([tc.wiggleshape for tc in f.tcal], dtype=np.float32)
ns.drs_mean_shape = np.array([tc.meanshape for tc in f.tcal], dtype=np.float32)

if "hf" in vars():
    hf.close()
    del hf

hf = h5py.File(hdf5_fnam, "w")

def cds(dsetname, comment):
    ds = hf.create_dataset(
        dsetname, 
        data=ns.__dict__[dsetname],
        shuffle=True,
        compression="gzip",
        compression_opts=1)
    ds.attrs["comment"] = comment

cds("drsraw", "raw uncorrected DRS samples [event][channel][sample]")
cds("drs", "DRS samples with pedestal and gain corrections applied")
cds("drsu", "DRS corrected samples, resampled to equal time intervals")
cds("traw", "nominal uncorrected DRS sample times")
cds("tcor", "DRS sample times, corrected via timing calibration")
cds("drs_trig_cell", "DRS trigger/stop cell ID [event][channel]")
cds("drs_tstamp", "CAEN board time stamp [event]")
cds("drs_cellgains", "voltage gain [channel][cell] from DRS calibration")
cds("drs_cellwidth", "cell width (ns) [channel][cell] from DRS timing calibration")
cds("drs_peds", "DRS pedestal [channel][cell] from DRS calibration")
cds("drs_wiggle_shape", "fitted 'wiggle' shape subtracted from each DRS waveform")
cds("drs_mean_shape", "mean waveform, in absence of signal, CAEN board artifact")
hf.close()
del hf
print(f"hdf5 elapsed time: {time.time()-t0:.1f}s")
print(f"total elapsed time {time.time()-starttime:.1f}s")

In [ ]:
# move to position 4
position = "04"
pico.exec("m.move_distance_mm_y(4, speed_fast=True, forward=True)")
pico.exec("m.move_distance_mm_x(4, speed_fast=True, forward=True)")

# acquire and plot
starttime = time.time()
nevents = 4000
t0 = time.time()
f.do_triggered_readout(nevents)
print(list(f.trigev[0].__dict__.keys()))
print(f"acquisition elapsed time {time.time()-t0:.1f}s")
print(f"rough rate = {nevents/(time.time()-t0):.1f}Hz")
beep(1)
t0 = time.time()
f.correct_triggered_readout()
print(f"corrections elapsed time {time.time()-t0:.1f}s")
beep(1)

for ich in [0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16]:
    print(f"SiPM Channel {ich+1} of 16")
    for ev in range (min(1000, nevents)):
        plt.plot(f.trigev[ev].drsu[ich])
    plt.show()
    print(f"----------------------")
    
t0 = time.time()
hdf5_fnam = f"20260511_hfv2_TAC1p9x19_44p5V_Ref100mV_Sum000mV_100e_pos{position}_CAEN.hdf5"

ns = SimpleNamespace()
ns.drsraw = np.array([e.drsraw for e in f.trigev], dtype=np.int16)
ns.drs = np.array([e.drsgc for e in f.trigev], dtype=np.float32)
ns.drsu = np.array([e.drsu for e in f.trigev], dtype=np.float32)
ns.traw = np.array([e.traw for e in f.trigev], dtype=np.float32)
ns.tcor = np.array([e.tcor for e in f.trigev], dtype=np.float32)
ns.drs_trig_cell = np.array([e.drs_trig_cell for e in f.trigev], dtype=np.int16)
ns.drs_tstamp = np.array([e.tstamp for e in f.trigev], dtype=np.int64)
ns.drs_cellgains = np.array([tc.cellgain for tc in f.tcal], dtype=np.float32)
ns.drs_cellwidth = np.array([tc.celldt for tc in f.tcal], dtype=np.float32)
ns.drs_peds = np.array([tc.cellpeds for tc in f.tcal], dtype=np.float32)
ns.drs_wiggle_shape = np.array([tc.wiggleshape for tc in f.tcal], dtype=np.float32)
ns.drs_mean_shape = np.array([tc.meanshape for tc in f.tcal], dtype=np.float32)

if "hf" in vars():
    hf.close()
    del hf

hf = h5py.File(hdf5_fnam, "w")

def cds(dsetname, comment):
    ds = hf.create_dataset(
        dsetname, 
        data=ns.__dict__[dsetname],
        shuffle=True,
        compression="gzip",
        compression_opts=1)
    ds.attrs["comment"] = comment

cds("drsraw", "raw uncorrected DRS samples [event][channel][sample]")
cds("drs", "DRS samples with pedestal and gain corrections applied")
cds("drsu", "DRS corrected samples, resampled to equal time intervals")
cds("traw", "nominal uncorrected DRS sample times")
cds("tcor", "DRS sample times, corrected via timing calibration")
cds("drs_trig_cell", "DRS trigger/stop cell ID [event][channel]")
cds("drs_tstamp", "CAEN board time stamp [event]")
cds("drs_cellgains", "voltage gain [channel][cell] from DRS calibration")
cds("drs_cellwidth", "cell width (ns) [channel][cell] from DRS timing calibration")
cds("drs_peds", "DRS pedestal [channel][cell] from DRS calibration")
cds("drs_wiggle_shape", "fitted 'wiggle' shape subtracted from each DRS waveform")
cds("drs_mean_shape", "mean waveform, in absence of signal, CAEN board artifact")
hf.close()
del hf
print(f"hdf5 elapsed time: {time.time()-t0:.1f}s")
print(f"total elapsed time {time.time()-starttime:.1f}s")

In [ ]:
pico.exec("m.move_distance_mm_y(16, speed_fast=True, forward=False)")
pico.exec("m.move_distance_mm_x(16, speed_fast=True, forward=False)")

In [ ]:
print(pico.exec("m.move_distance_mm_y(4, speed_fast=True, forward=True)"))
print(pico.exec("m.move_distance_mm_x(4, speed_fast=True, forward=False)"))

In [ ]:
print(pico.exec("m.move_distance_mm_x(4, speed_fast=True, forward=False)"))

In [ ]:
starttime = time.time()
nevents = 100
t0 = time.time()
f.do_triggered_readout(nevents)
print(list(f.trigev[0].__dict__.keys()))
print(f"acquisition elapsed time {time.time()-t0:.1f}s")
print(f"rough rate = {nevents/(time.time()-t0):.1f}Hz")
beep(1)
t0 = time.time()
f.correct_triggered_readout()
print(f"corrections elapsed time {time.time()-t0:.1f}s")
beep(1)

for ich in [0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16]:
    print(f"SiPM Channel {ich+1} of 16")
    for ev in range (min(1000, nevents)):
        plt.plot(f.trigev[ev].drsu[ich])
    plt.show()
    print(f"----------------------")
    
t0 = time.time()
hdf5_fnam = "20260527_hfv2_TAC1p9x19_44p5V_Ref100mV_Sum000mV_100e_single_CAEN.hdf5"

ns = SimpleNamespace()
ns.drsraw = np.array([e.drsraw for e in f.trigev], dtype=np.int16)
ns.drs = np.array([e.drsgc for e in f.trigev], dtype=np.float32)
ns.drsu = np.array([e.drsu for e in f.trigev], dtype=np.float32)
ns.traw = np.array([e.traw for e in f.trigev], dtype=np.float32)
ns.tcor = np.array([e.tcor for e in f.trigev], dtype=np.float32)
ns.drs_trig_cell = np.array([e.drs_trig_cell for e in f.trigev], dtype=np.int16)
ns.drs_tstamp = np.array([e.tstamp for e in f.trigev], dtype=np.int64)
ns.drs_cellgains = np.array([tc.cellgain for tc in f.tcal], dtype=np.float32)
ns.drs_cellwidth = np.array([tc.celldt for tc in f.tcal], dtype=np.float32)
ns.drs_peds = np.array([tc.cellpeds for tc in f.tcal], dtype=np.float32)
ns.drs_wiggle_shape = np.array([tc.wiggleshape for tc in f.tcal], dtype=np.float32)
ns.drs_mean_shape = np.array([tc.meanshape for tc in f.tcal], dtype=np.float32)

if "hf" in vars():
    hf.close()
    del hf

hf = h5py.File(hdf5_fnam, "w")

def cds(dsetname, comment):
    ds = hf.create_dataset(
        dsetname, 
        data=ns.__dict__[dsetname],
        shuffle=True,
        compression="gzip",
        compression_opts=1)
    ds.attrs["comment"] = comment

cds("drsraw", "raw uncorrected DRS samples [event][channel][sample]")
cds("drs", "DRS samples with pedestal and gain corrections applied")
cds("drsu", "DRS corrected samples, resampled to equal time intervals")
cds("traw", "nominal uncorrected DRS sample times")
cds("tcor", "DRS sample times, corrected via timing calibration")
cds("drs_trig_cell", "DRS trigger/stop cell ID [event][channel]")
cds("drs_tstamp", "CAEN board time stamp [event]")
cds("drs_cellgains", "voltage gain [channel][cell] from DRS calibration")
cds("drs_cellwidth", "cell width (ns) [channel][cell] from DRS timing calibration")
cds("drs_peds", "DRS pedestal [channel][cell] from DRS calibration")
cds("drs_wiggle_shape", "fitted 'wiggle' shape subtracted from each DRS waveform")
cds("drs_mean_shape", "mean waveform, in absence of signal, CAEN board artifact")
hf.close()
del hf
print(f"hdf5 elapsed time: {time.time()-t0:.1f}s")
print(f"total elapsed time {time.time()-starttime:.1f}s")

In [ ]:
starttime = time.time()
nevents = 10000
t0 = time.time()
f.do_triggered_readout(nevents)
print(list(f.trigev[0].__dict__.keys()))
print(f"acquisition elapsed time {time.time()-t0:.1f}s")
print(f"rough rate = {nevents/(time.time()-t0):.1f}Hz")
beep(1)
t0 = time.time()
f.correct_triggered_readout()
print(f"corrections elapsed time {time.time()-t0:.1f}s")
beep(1)

t0 = time.time()
hdf5_fnam = "20260123_nuvmtarray_TAC1p9x19_47p5V_Ref280mV_Sum400mV_DB1_midleft3x2_10ke_coinc_CAEN_offset1mm.hdf5"

ns = SimpleNamespace()
ns.drsraw = np.array([e.drsraw for e in f.trigev], dtype=np.int16)
ns.drs = np.array([e.drsgc for e in f.trigev], dtype=np.float32)
ns.drsu = np.array([e.drsu for e in f.trigev], dtype=np.float32)
ns.traw = np.array([e.traw for e in f.trigev], dtype=np.float32)
ns.tcor = np.array([e.tcor for e in f.trigev], dtype=np.float32)
ns.drs_trig_cell = np.array([e.drs_trig_cell for e in f.trigev], dtype=np.int16)
ns.drs_tstamp = np.array([e.tstamp for e in f.trigev], dtype=np.int64)
ns.drs_cellgains = np.array([tc.cellgain for tc in f.tcal], dtype=np.float32)
ns.drs_cellwidth = np.array([tc.celldt for tc in f.tcal], dtype=np.float32)
ns.drs_peds = np.array([tc.cellpeds for tc in f.tcal], dtype=np.float32)
ns.drs_wiggle_shape = np.array([tc.wiggleshape for tc in f.tcal], dtype=np.float32)
ns.drs_mean_shape = np.array([tc.meanshape for tc in f.tcal], dtype=np.float32)

if "hf" in vars():
    hf.close()
    del hf

hf = h5py.File(hdf5_fnam, "w")

def cds(dsetname, comment):
    ds = hf.create_dataset(
        dsetname, 
        data=ns.__dict__[dsetname],
        shuffle=True,
        compression="gzip",
        compression_opts=1)
    ds.attrs["comment"] = comment

cds("drsraw", "raw uncorrected DRS samples [event][channel][sample]")
cds("drs", "DRS samples with pedestal and gain corrections applied")
cds("drsu", "DRS corrected samples, resampled to equal time intervals")
cds("traw", "nominal uncorrected DRS sample times")
cds("tcor", "DRS sample times, corrected via timing calibration")
cds("drs_trig_cell", "DRS trigger/stop cell ID [event][channel]")
cds("drs_tstamp", "CAEN board time stamp [event]")
cds("drs_cellgains", "voltage gain [channel][cell] from DRS calibration")
cds("drs_cellwidth", "cell width (ns) [channel][cell] from DRS timing calibration")
cds("drs_peds", "DRS pedestal [channel][cell] from DRS calibration")
cds("drs_wiggle_shape", "fitted 'wiggle' shape subtracted from each DRS waveform")
cds("drs_mean_shape", "mean waveform, in absence of signal, CAEN board artifact")
hf.close()
del hf
print(f"hdf5 elapsed time: {time.time()-t0:.1f}s")
print(f"total elapsed time {time.time()-starttime:.1f}s")


In [ ]:
starttime = time.time()
nevents = 10000
t0 = time.time()
f.do_triggered_readout(nevents)
print(list(f.trigev[0].__dict__.keys()))
print(f"acquisition elapsed time {time.time()-t0:.1f}s")
print(f"rough rate = {nevents/(time.time()-t0):.1f}Hz")
beep(1)
t0 = time.time()
f.correct_triggered_readout()
print(f"corrections elapsed time {time.time()-t0:.1f}s")
beep(1)

t0 = time.time()
hdf5_fnam = "20260123_nuvmtarray_TAC1p9x19_47p5V_Ref280mV_Sum420mV_DB1_midleft3x2_10ke_coinc_CAEN_offset2mm.hdf5"

ns = SimpleNamespace()
ns.drsraw = np.array([e.drsraw for e in f.trigev], dtype=np.int16)
ns.drs = np.array([e.drsgc for e in f.trigev], dtype=np.float32)
ns.drsu = np.array([e.drsu for e in f.trigev], dtype=np.float32)
ns.traw = np.array([e.traw for e in f.trigev], dtype=np.float32)
ns.tcor = np.array([e.tcor for e in f.trigev], dtype=np.float32)
ns.drs_trig_cell = np.array([e.drs_trig_cell for e in f.trigev], dtype=np.int16)
ns.drs_tstamp = np.array([e.tstamp for e in f.trigev], dtype=np.int64)
ns.drs_cellgains = np.array([tc.cellgain for tc in f.tcal], dtype=np.float32)
ns.drs_cellwidth = np.array([tc.celldt for tc in f.tcal], dtype=np.float32)
ns.drs_peds = np.array([tc.cellpeds for tc in f.tcal], dtype=np.float32)
ns.drs_wiggle_shape = np.array([tc.wiggleshape for tc in f.tcal], dtype=np.float32)
ns.drs_mean_shape = np.array([tc.meanshape for tc in f.tcal], dtype=np.float32)

if "hf" in vars():
    hf.close()
    del hf

hf = h5py.File(hdf5_fnam, "w")

def cds(dsetname, comment):
    ds = hf.create_dataset(
        dsetname, 
        data=ns.__dict__[dsetname],
        shuffle=True,
        compression="gzip",
        compression_opts=1)
    ds.attrs["comment"] = comment

cds("drsraw", "raw uncorrected DRS samples [event][channel][sample]")
cds("drs", "DRS samples with pedestal and gain corrections applied")
cds("drsu", "DRS corrected samples, resampled to equal time intervals")
cds("traw", "nominal uncorrected DRS sample times")
cds("tcor", "DRS sample times, corrected via timing calibration")
cds("drs_trig_cell", "DRS trigger/stop cell ID [event][channel]")
cds("drs_tstamp", "CAEN board time stamp [event]")
cds("drs_cellgains", "voltage gain [channel][cell] from DRS calibration")
cds("drs_cellwidth", "cell width (ns) [channel][cell] from DRS timing calibration")
cds("drs_peds", "DRS pedestal [channel][cell] from DRS calibration")
cds("drs_wiggle_shape", "fitted 'wiggle' shape subtracted from each DRS waveform")
cds("drs_mean_shape", "mean waveform, in absence of signal, CAEN board artifact")
hf.close()
del hf
print(f"hdf5 elapsed time: {time.time()-t0:.1f}s")
print(f"total elapsed time {time.time()-starttime:.1f}s")

In [ ]:
starttime = time.time()
nevents = 10000
t0 = time.time()
f.do_triggered_readout(nevents)
print(list(f.trigev[0].__dict__.keys()))
print(f"acquisition elapsed time {time.time()-t0:.1f}s")
print(f"rough rate = {nevents/(time.time()-t0):.1f}Hz")
beep(1)
t0 = time.time()
f.correct_triggered_readout()
print(f"corrections elapsed time {time.time()-t0:.1f}s")
beep(1)

t0 = time.time()
hdf5_fnam = "20260123_nuvmtarray_TAC1p9x19_47p5V_Ref280mV_Sum410mV_DB1_midleft3x2_10ke_coinc_CAEN_offset3mm.hdf5"

ns = SimpleNamespace()
ns.drsraw = np.array([e.drsraw for e in f.trigev], dtype=np.int16)
ns.drs = np.array([e.drsgc for e in f.trigev], dtype=np.float32)
ns.drsu = np.array([e.drsu for e in f.trigev], dtype=np.float32)
ns.traw = np.array([e.traw for e in f.trigev], dtype=np.float32)
ns.tcor = np.array([e.tcor for e in f.trigev], dtype=np.float32)
ns.drs_trig_cell = np.array([e.drs_trig_cell for e in f.trigev], dtype=np.int16)
ns.drs_tstamp = np.array([e.tstamp for e in f.trigev], dtype=np.int64)
ns.drs_cellgains = np.array([tc.cellgain for tc in f.tcal], dtype=np.float32)
ns.drs_cellwidth = np.array([tc.celldt for tc in f.tcal], dtype=np.float32)
ns.drs_peds = np.array([tc.cellpeds for tc in f.tcal], dtype=np.float32)
ns.drs_wiggle_shape = np.array([tc.wiggleshape for tc in f.tcal], dtype=np.float32)
ns.drs_mean_shape = np.array([tc.meanshape for tc in f.tcal], dtype=np.float32)

if "hf" in vars():
    hf.close()
    del hf

hf = h5py.File(hdf5_fnam, "w")

def cds(dsetname, comment):
    ds = hf.create_dataset(
        dsetname, 
        data=ns.__dict__[dsetname],
        shuffle=True,
        compression="gzip",
        compression_opts=1)
    ds.attrs["comment"] = comment

cds("drsraw", "raw uncorrected DRS samples [event][channel][sample]")
cds("drs", "DRS samples with pedestal and gain corrections applied")
cds("drsu", "DRS corrected samples, resampled to equal time intervals")
cds("traw", "nominal uncorrected DRS sample times")
cds("tcor", "DRS sample times, corrected via timing calibration")
cds("drs_trig_cell", "DRS trigger/stop cell ID [event][channel]")
cds("drs_tstamp", "CAEN board time stamp [event]")
cds("drs_cellgains", "voltage gain [channel][cell] from DRS calibration")
cds("drs_cellwidth", "cell width (ns) [channel][cell] from DRS timing calibration")
cds("drs_peds", "DRS pedestal [channel][cell] from DRS calibration")
cds("drs_wiggle_shape", "fitted 'wiggle' shape subtracted from each DRS waveform")
cds("drs_mean_shape", "mean waveform, in absence of signal, CAEN board artifact")
hf.close()
del hf
print(f"hdf5 elapsed time: {time.time()-t0:.1f}s")
print(f"total elapsed time {time.time()-starttime:.1f}s")

In [ ]:
starttime = time.time()
nevents = 10000
t0 = time.time()
f.do_triggered_readout(nevents)
print(list(f.trigev[0].__dict__.keys()))
print(f"acquisition elapsed time {time.time()-t0:.1f}s")
print(f"rough rate = {nevents/(time.time()-t0):.1f}Hz")
beep(1)
t0 = time.time()
f.correct_triggered_readout()
print(f"corrections elapsed time {time.time()-t0:.1f}s")
beep(1)

t0 = time.time()
hdf5_fnam = "20260123_nuvmtarray_TAC1p9x19_47p5V_Ref280mV_Sum400mV_DB1_midleft3x2_10ke_coinc_CAEN_offset4mm.hdf5"

ns = SimpleNamespace()
ns.drsraw = np.array([e.drsraw for e in f.trigev], dtype=np.int16)
ns.drs = np.array([e.drsgc for e in f.trigev], dtype=np.float32)
ns.drsu = np.array([e.drsu for e in f.trigev], dtype=np.float32)
ns.traw = np.array([e.traw for e in f.trigev], dtype=np.float32)
ns.tcor = np.array([e.tcor for e in f.trigev], dtype=np.float32)
ns.drs_trig_cell = np.array([e.drs_trig_cell for e in f.trigev], dtype=np.int16)
ns.drs_tstamp = np.array([e.tstamp for e in f.trigev], dtype=np.int64)
ns.drs_cellgains = np.array([tc.cellgain for tc in f.tcal], dtype=np.float32)
ns.drs_cellwidth = np.array([tc.celldt for tc in f.tcal], dtype=np.float32)
ns.drs_peds = np.array([tc.cellpeds for tc in f.tcal], dtype=np.float32)
ns.drs_wiggle_shape = np.array([tc.wiggleshape for tc in f.tcal], dtype=np.float32)
ns.drs_mean_shape = np.array([tc.meanshape for tc in f.tcal], dtype=np.float32)

if "hf" in vars():
    hf.close()
    del hf

hf = h5py.File(hdf5_fnam, "w")

def cds(dsetname, comment):
    ds = hf.create_dataset(
        dsetname, 
        data=ns.__dict__[dsetname],
        shuffle=True,
        compression="gzip",
        compression_opts=1)
    ds.attrs["comment"] = comment

cds("drsraw", "raw uncorrected DRS samples [event][channel][sample]")
cds("drs", "DRS samples with pedestal and gain corrections applied")
cds("drsu", "DRS corrected samples, resampled to equal time intervals")
cds("traw", "nominal uncorrected DRS sample times")
cds("tcor", "DRS sample times, corrected via timing calibration")
cds("drs_trig_cell", "DRS trigger/stop cell ID [event][channel]")
cds("drs_tstamp", "CAEN board time stamp [event]")
cds("drs_cellgains", "voltage gain [channel][cell] from DRS calibration")
cds("drs_cellwidth", "cell width (ns) [channel][cell] from DRS timing calibration")
cds("drs_peds", "DRS pedestal [channel][cell] from DRS calibration")
cds("drs_wiggle_shape", "fitted 'wiggle' shape subtracted from each DRS waveform")
cds("drs_mean_shape", "mean waveform, in absence of signal, CAEN board artifact")
hf.close()
del hf
print(f"hdf5 elapsed time: {time.time()-t0:.1f}s")
print(f"total elapsed time {time.time()-starttime:.1f}s")

In [ ]:
starttime = time.time()
nevents = 10000
t0 = time.time()
f.do_triggered_readout(nevents)
print(list(f.trigev[0].__dict__.keys()))
print(f"acquisition elapsed time {time.time()-t0:.1f}s")
print(f"rough rate = {nevents/(time.time()-t0):.1f}Hz")
beep(1)
t0 = time.time()
f.correct_triggered_readout()
print(f"corrections elapsed time {time.time()-t0:.1f}s")
beep(1)

t0 = time.time()
hdf5_fnam = "20250625_nuvmtarray_4x4xtal_47p5V_Ref250mV_Sum400mV_DB1_midleft3x2_15ke_coinc_CAEN_run06.hdf5"

ns = SimpleNamespace()
ns.drsraw = np.array([e.drsraw for e in f.trigev], dtype=np.int16)
ns.drs = np.array([e.drsgc for e in f.trigev], dtype=np.float32)
ns.drsu = np.array([e.drsu for e in f.trigev], dtype=np.float32)
ns.traw = np.array([e.traw for e in f.trigev], dtype=np.float32)
ns.tcor = np.array([e.tcor for e in f.trigev], dtype=np.float32)
ns.drs_trig_cell = np.array([e.drs_trig_cell for e in f.trigev], dtype=np.int16)
ns.drs_tstamp = np.array([e.tstamp for e in f.trigev], dtype=np.int64)
ns.drs_cellgains = np.array([tc.cellgain for tc in f.tcal], dtype=np.float32)
ns.drs_cellwidth = np.array([tc.celldt for tc in f.tcal], dtype=np.float32)
ns.drs_peds = np.array([tc.cellpeds for tc in f.tcal], dtype=np.float32)
ns.drs_wiggle_shape = np.array([tc.wiggleshape for tc in f.tcal], dtype=np.float32)
ns.drs_mean_shape = np.array([tc.meanshape for tc in f.tcal], dtype=np.float32)

if "hf" in vars():
    hf.close()
    del hf

hf = h5py.File(hdf5_fnam, "w")

def cds(dsetname, comment):
    ds = hf.create_dataset(
        dsetname, 
        data=ns.__dict__[dsetname],
        shuffle=True,
        compression="gzip",
        compression_opts=1)
    ds.attrs["comment"] = comment

cds("drsraw", "raw uncorrected DRS samples [event][channel][sample]")
cds("drs", "DRS samples with pedestal and gain corrections applied")
cds("drsu", "DRS corrected samples, resampled to equal time intervals")
cds("traw", "nominal uncorrected DRS sample times")
cds("tcor", "DRS sample times, corrected via timing calibration")
cds("drs_trig_cell", "DRS trigger/stop cell ID [event][channel]")
cds("drs_tstamp", "CAEN board time stamp [event]")
cds("drs_cellgains", "voltage gain [channel][cell] from DRS calibration")
cds("drs_cellwidth", "cell width (ns) [channel][cell] from DRS timing calibration")
cds("drs_peds", "DRS pedestal [channel][cell] from DRS calibration")
cds("drs_wiggle_shape", "fitted 'wiggle' shape subtracted from each DRS waveform")
cds("drs_mean_shape", "mean waveform, in absence of signal, CAEN board artifact")
hf.close()
del hf
print(f"hdf5 elapsed time: {time.time()-t0:.1f}s")
print(f"total elapsed time {time.time()-starttime:.1f}s")

In [ ]:
starttime = time.time()
nevents = 15000
t0 = time.time()
f.do_triggered_readout(nevents)
print(list(f.trigev[0].__dict__.keys()))
print(f"acquisition elapsed time {time.time()-t0:.1f}s")
print(f"rough rate = {nevents/(time.time()-t0):.1f}Hz")
beep(1)
t0 = time.time()
f.correct_triggered_readout()
print(f"corrections elapsed time {time.time()-t0:.1f}s")
beep(1)

t0 = time.time()
hdf5_fnam = "20250625_nuvmtarray_4x4xtal_47p5V_Ref250mV_Sum400mV_DB1_midleft3x2_15ke_coinc_CAEN_run07.hdf5"

ns = SimpleNamespace()
ns.drsraw = np.array([e.drsraw for e in f.trigev], dtype=np.int16)
ns.drs = np.array([e.drsgc for e in f.trigev], dtype=np.float32)
ns.drsu = np.array([e.drsu for e in f.trigev], dtype=np.float32)
ns.traw = np.array([e.traw for e in f.trigev], dtype=np.float32)
ns.tcor = np.array([e.tcor for e in f.trigev], dtype=np.float32)
ns.drs_trig_cell = np.array([e.drs_trig_cell for e in f.trigev], dtype=np.int16)
ns.drs_tstamp = np.array([e.tstamp for e in f.trigev], dtype=np.int64)
ns.drs_cellgains = np.array([tc.cellgain for tc in f.tcal], dtype=np.float32)
ns.drs_cellwidth = np.array([tc.celldt for tc in f.tcal], dtype=np.float32)
ns.drs_peds = np.array([tc.cellpeds for tc in f.tcal], dtype=np.float32)
ns.drs_wiggle_shape = np.array([tc.wiggleshape for tc in f.tcal], dtype=np.float32)
ns.drs_mean_shape = np.array([tc.meanshape for tc in f.tcal], dtype=np.float32)

if "hf" in vars():
    hf.close()
    del hf

hf = h5py.File(hdf5_fnam, "w")

def cds(dsetname, comment):
    ds = hf.create_dataset(
        dsetname, 
        data=ns.__dict__[dsetname],
        shuffle=True,
        compression="gzip",
        compression_opts=1)
    ds.attrs["comment"] = comment

cds("drsraw", "raw uncorrected DRS samples [event][channel][sample]")
cds("drs", "DRS samples with pedestal and gain corrections applied")
cds("drsu", "DRS corrected samples, resampled to equal time intervals")
cds("traw", "nominal uncorrected DRS sample times")
cds("tcor", "DRS sample times, corrected via timing calibration")
cds("drs_trig_cell", "DRS trigger/stop cell ID [event][channel]")
cds("drs_tstamp", "CAEN board time stamp [event]")
cds("drs_cellgains", "voltage gain [channel][cell] from DRS calibration")
cds("drs_cellwidth", "cell width (ns) [channel][cell] from DRS timing calibration")
cds("drs_peds", "DRS pedestal [channel][cell] from DRS calibration")
cds("drs_wiggle_shape", "fitted 'wiggle' shape subtracted from each DRS waveform")
cds("drs_mean_shape", "mean waveform, in absence of signal, CAEN board artifact")
hf.close()
del hf
print(f"hdf5 elapsed time: {time.time()-t0:.1f}s")
print(f"total elapsed time {time.time()-starttime:.1f}s")

In [ ]:
starttime = time.time()
nevents = 15000
t0 = time.time()
f.do_triggered_readout(nevents)
print(list(f.trigev[0].__dict__.keys()))
print(f"acquisition elapsed time {time.time()-t0:.1f}s")
print(f"rough rate = {nevents/(time.time()-t0):.1f}Hz")
beep(1)
t0 = time.time()
f.correct_triggered_readout()
print(f"corrections elapsed time {time.time()-t0:.1f}s")
beep(1)

t0 = time.time()
hdf5_fnam = "20250625_nuvmtarray_4x4xtal_47p5V_Ref250mV_Sum400mV_DB1_midleft3x2_15ke_coinc_CAEN_run08.hdf5"

ns = SimpleNamespace()
ns.drsraw = np.array([e.drsraw for e in f.trigev], dtype=np.int16)
ns.drs = np.array([e.drsgc for e in f.trigev], dtype=np.float32)
ns.drsu = np.array([e.drsu for e in f.trigev], dtype=np.float32)
ns.traw = np.array([e.traw for e in f.trigev], dtype=np.float32)
ns.tcor = np.array([e.tcor for e in f.trigev], dtype=np.float32)
ns.drs_trig_cell = np.array([e.drs_trig_cell for e in f.trigev], dtype=np.int16)
ns.drs_tstamp = np.array([e.tstamp for e in f.trigev], dtype=np.int64)
ns.drs_cellgains = np.array([tc.cellgain for tc in f.tcal], dtype=np.float32)
ns.drs_cellwidth = np.array([tc.celldt for tc in f.tcal], dtype=np.float32)
ns.drs_peds = np.array([tc.cellpeds for tc in f.tcal], dtype=np.float32)
ns.drs_wiggle_shape = np.array([tc.wiggleshape for tc in f.tcal], dtype=np.float32)
ns.drs_mean_shape = np.array([tc.meanshape for tc in f.tcal], dtype=np.float32)

if "hf" in vars():
    hf.close()
    del hf

hf = h5py.File(hdf5_fnam, "w")

def cds(dsetname, comment):
    ds = hf.create_dataset(
        dsetname, 
        data=ns.__dict__[dsetname],
        shuffle=True,
        compression="gzip",
        compression_opts=1)
    ds.attrs["comment"] = comment

cds("drsraw", "raw uncorrected DRS samples [event][channel][sample]")
cds("drs", "DRS samples with pedestal and gain corrections applied")
cds("drsu", "DRS corrected samples, resampled to equal time intervals")
cds("traw", "nominal uncorrected DRS sample times")
cds("tcor", "DRS sample times, corrected via timing calibration")
cds("drs_trig_cell", "DRS trigger/stop cell ID [event][channel]")
cds("drs_tstamp", "CAEN board time stamp [event]")
cds("drs_cellgains", "voltage gain [channel][cell] from DRS calibration")
cds("drs_cellwidth", "cell width (ns) [channel][cell] from DRS timing calibration")
cds("drs_peds", "DRS pedestal [channel][cell] from DRS calibration")
cds("drs_wiggle_shape", "fitted 'wiggle' shape subtracted from each DRS waveform")
cds("drs_mean_shape", "mean waveform, in absence of signal, CAEN board artifact")
hf.close()
del hf
print(f"hdf5 elapsed time: {time.time()-t0:.1f}s")
print(f"total elapsed time {time.time()-starttime:.1f}s")

In [ ]:
starttime = time.time()
nevents = 15000
t0 = time.time()
f.do_triggered_readout(nevents)
print(list(f.trigev[0].__dict__.keys()))
print(f"acquisition elapsed time {time.time()-t0:.1f}s")
print(f"rough rate = {nevents/(time.time()-t0):.1f}Hz")
beep(1)
t0 = time.time()
f.correct_triggered_readout()
print(f"corrections elapsed time {time.time()-t0:.1f}s")
beep(1)

t0 = time.time()
hdf5_fnam = "20250625_nuvmtarray_4x4xtal_47p5V_Ref250mV_Sum400mV_DB1_midleft3x2_15ke_coinc_CAEN_run09.hdf5"

ns = SimpleNamespace()
ns.drsraw = np.array([e.drsraw for e in f.trigev], dtype=np.int16)
ns.drs = np.array([e.drsgc for e in f.trigev], dtype=np.float32)
ns.drsu = np.array([e.drsu for e in f.trigev], dtype=np.float32)
ns.traw = np.array([e.traw for e in f.trigev], dtype=np.float32)
ns.tcor = np.array([e.tcor for e in f.trigev], dtype=np.float32)
ns.drs_trig_cell = np.array([e.drs_trig_cell for e in f.trigev], dtype=np.int16)
ns.drs_tstamp = np.array([e.tstamp for e in f.trigev], dtype=np.int64)
ns.drs_cellgains = np.array([tc.cellgain for tc in f.tcal], dtype=np.float32)
ns.drs_cellwidth = np.array([tc.celldt for tc in f.tcal], dtype=np.float32)
ns.drs_peds = np.array([tc.cellpeds for tc in f.tcal], dtype=np.float32)
ns.drs_wiggle_shape = np.array([tc.wiggleshape for tc in f.tcal], dtype=np.float32)
ns.drs_mean_shape = np.array([tc.meanshape for tc in f.tcal], dtype=np.float32)

if "hf" in vars():
    hf.close()
    del hf

hf = h5py.File(hdf5_fnam, "w")

def cds(dsetname, comment):
    ds = hf.create_dataset(
        dsetname, 
        data=ns.__dict__[dsetname],
        shuffle=True,
        compression="gzip",
        compression_opts=1)
    ds.attrs["comment"] = comment

cds("drsraw", "raw uncorrected DRS samples [event][channel][sample]")
cds("drs", "DRS samples with pedestal and gain corrections applied")
cds("drsu", "DRS corrected samples, resampled to equal time intervals")
cds("traw", "nominal uncorrected DRS sample times")
cds("tcor", "DRS sample times, corrected via timing calibration")
cds("drs_trig_cell", "DRS trigger/stop cell ID [event][channel]")
cds("drs_tstamp", "CAEN board time stamp [event]")
cds("drs_cellgains", "voltage gain [channel][cell] from DRS calibration")
cds("drs_cellwidth", "cell width (ns) [channel][cell] from DRS timing calibration")
cds("drs_peds", "DRS pedestal [channel][cell] from DRS calibration")
cds("drs_wiggle_shape", "fitted 'wiggle' shape subtracted from each DRS waveform")
cds("drs_mean_shape", "mean waveform, in absence of signal, CAEN board artifact")
hf.close()
del hf
print(f"hdf5 elapsed time: {time.time()-t0:.1f}s")
print(f"total elapsed time {time.time()-starttime:.1f}s")

In [ ]:
starttime = time.time()
nevents = 15000
t0 = time.time()
f.do_triggered_readout(nevents)
print(list(f.trigev[0].__dict__.keys()))
print(f"acquisition elapsed time {time.time()-t0:.1f}s")
print(f"rough rate = {nevents/(time.time()-t0):.1f}Hz")
beep(1)
t0 = time.time()
f.correct_triggered_readout()
print(f"corrections elapsed time {time.time()-t0:.1f}s")
beep(1)

t0 = time.time()
hdf5_fnam = "20250625_nuvmtarray_4x4xtal_47p5V_Ref250mV_Sum400mV_DB1_midleft3x2_15ke_coinc_CAEN_run10.hdf5"

ns = SimpleNamespace()
ns.drsraw = np.array([e.drsraw for e in f.trigev], dtype=np.int16)
ns.drs = np.array([e.drsgc for e in f.trigev], dtype=np.float32)
ns.drsu = np.array([e.drsu for e in f.trigev], dtype=np.float32)
ns.traw = np.array([e.traw for e in f.trigev], dtype=np.float32)
ns.tcor = np.array([e.tcor for e in f.trigev], dtype=np.float32)
ns.drs_trig_cell = np.array([e.drs_trig_cell for e in f.trigev], dtype=np.int16)
ns.drs_tstamp = np.array([e.tstamp for e in f.trigev], dtype=np.int64)
ns.drs_cellgains = np.array([tc.cellgain for tc in f.tcal], dtype=np.float32)
ns.drs_cellwidth = np.array([tc.celldt for tc in f.tcal], dtype=np.float32)
ns.drs_peds = np.array([tc.cellpeds for tc in f.tcal], dtype=np.float32)
ns.drs_wiggle_shape = np.array([tc.wiggleshape for tc in f.tcal], dtype=np.float32)
ns.drs_mean_shape = np.array([tc.meanshape for tc in f.tcal], dtype=np.float32)

if "hf" in vars():
    hf.close()
    del hf

hf = h5py.File(hdf5_fnam, "w")

def cds(dsetname, comment):
    ds = hf.create_dataset(
        dsetname, 
        data=ns.__dict__[dsetname],
        shuffle=True,
        compression="gzip",
        compression_opts=1)
    ds.attrs["comment"] = comment

cds("drsraw", "raw uncorrected DRS samples [event][channel][sample]")
cds("drs", "DRS samples with pedestal and gain corrections applied")
cds("drsu", "DRS corrected samples, resampled to equal time intervals")
cds("traw", "nominal uncorrected DRS sample times")
cds("tcor", "DRS sample times, corrected via timing calibration")
cds("drs_trig_cell", "DRS trigger/stop cell ID [event][channel]")
cds("drs_tstamp", "CAEN board time stamp [event]")
cds("drs_cellgains", "voltage gain [channel][cell] from DRS calibration")
cds("drs_cellwidth", "cell width (ns) [channel][cell] from DRS timing calibration")
cds("drs_peds", "DRS pedestal [channel][cell] from DRS calibration")
cds("drs_wiggle_shape", "fitted 'wiggle' shape subtracted from each DRS waveform")
cds("drs_mean_shape", "mean waveform, in absence of signal, CAEN board artifact")
hf.close()
del hf
print(f"hdf5 elapsed time: {time.time()-t0:.1f}s")
print(f"total elapsed time {time.time()-starttime:.1f}s")

In [ ]:
starttime = time.time()
nevents = 15000
t0 = time.time()
f.do_triggered_readout(nevents)
print(list(f.trigev[0].__dict__.keys()))
print(f"acquisition elapsed time {time.time()-t0:.1f}s")
print(f"rough rate = {nevents/(time.time()-t0):.1f}Hz")
beep(1)
t0 = time.time()
f.correct_triggered_readout()
print(f"corrections elapsed time {time.time()-t0:.1f}s")
beep(1)

t0 = time.time()
hdf5_fnam = "20250625_nuvmtarray_4x4xtal_47p5V_Ref250mV_Sum400mV_DB1_midleft3x2_15ke_coinc_CAEN_run11.hdf5"

ns = SimpleNamespace()
ns.drsraw = np.array([e.drsraw for e in f.trigev], dtype=np.int16)
ns.drs = np.array([e.drsgc for e in f.trigev], dtype=np.float32)
ns.drsu = np.array([e.drsu for e in f.trigev], dtype=np.float32)
ns.traw = np.array([e.traw for e in f.trigev], dtype=np.float32)
ns.tcor = np.array([e.tcor for e in f.trigev], dtype=np.float32)
ns.drs_trig_cell = np.array([e.drs_trig_cell for e in f.trigev], dtype=np.int16)
ns.drs_tstamp = np.array([e.tstamp for e in f.trigev], dtype=np.int64)
ns.drs_cellgains = np.array([tc.cellgain for tc in f.tcal], dtype=np.float32)
ns.drs_cellwidth = np.array([tc.celldt for tc in f.tcal], dtype=np.float32)
ns.drs_peds = np.array([tc.cellpeds for tc in f.tcal], dtype=np.float32)
ns.drs_wiggle_shape = np.array([tc.wiggleshape for tc in f.tcal], dtype=np.float32)
ns.drs_mean_shape = np.array([tc.meanshape for tc in f.tcal], dtype=np.float32)

if "hf" in vars():
    hf.close()
    del hf

hf = h5py.File(hdf5_fnam, "w")

def cds(dsetname, comment):
    ds = hf.create_dataset(
        dsetname, 
        data=ns.__dict__[dsetname],
        shuffle=True,
        compression="gzip",
        compression_opts=1)
    ds.attrs["comment"] = comment

cds("drsraw", "raw uncorrected DRS samples [event][channel][sample]")
cds("drs", "DRS samples with pedestal and gain corrections applied")
cds("drsu", "DRS corrected samples, resampled to equal time intervals")
cds("traw", "nominal uncorrected DRS sample times")
cds("tcor", "DRS sample times, corrected via timing calibration")
cds("drs_trig_cell", "DRS trigger/stop cell ID [event][channel]")
cds("drs_tstamp", "CAEN board time stamp [event]")
cds("drs_cellgains", "voltage gain [channel][cell] from DRS calibration")
cds("drs_cellwidth", "cell width (ns) [channel][cell] from DRS timing calibration")
cds("drs_peds", "DRS pedestal [channel][cell] from DRS calibration")
cds("drs_wiggle_shape", "fitted 'wiggle' shape subtracted from each DRS waveform")
cds("drs_mean_shape", "mean waveform, in absence of signal, CAEN board artifact")
hf.close()
del hf
print(f"hdf5 elapsed time: {time.time()-t0:.1f}s")
print(f"total elapsed time {time.time()-starttime:.1f}s")

In [ ]:
starttime = time.time()
nevents = 15000
t0 = time.time()
f.do_triggered_readout(nevents)
print(list(f.trigev[0].__dict__.keys()))
print(f"acquisition elapsed time {time.time()-t0:.1f}s")
print(f"rough rate = {nevents/(time.time()-t0):.1f}Hz")
beep(1)
t0 = time.time()
f.correct_triggered_readout()
print(f"corrections elapsed time {time.time()-t0:.1f}s")
beep(1)

t0 = time.time()
hdf5_fnam = "20250625_nuvmtarray_4x4xtal_47p5V_Ref250mV_Sum400mV_DB1_midleft3x2_15ke_coinc_CAEN_run12.hdf5"

ns = SimpleNamespace()
ns.drsraw = np.array([e.drsraw for e in f.trigev], dtype=np.int16)
ns.drs = np.array([e.drsgc for e in f.trigev], dtype=np.float32)
ns.drsu = np.array([e.drsu for e in f.trigev], dtype=np.float32)
ns.traw = np.array([e.traw for e in f.trigev], dtype=np.float32)
ns.tcor = np.array([e.tcor for e in f.trigev], dtype=np.float32)
ns.drs_trig_cell = np.array([e.drs_trig_cell for e in f.trigev], dtype=np.int16)
ns.drs_tstamp = np.array([e.tstamp for e in f.trigev], dtype=np.int64)
ns.drs_cellgains = np.array([tc.cellgain for tc in f.tcal], dtype=np.float32)
ns.drs_cellwidth = np.array([tc.celldt for tc in f.tcal], dtype=np.float32)
ns.drs_peds = np.array([tc.cellpeds for tc in f.tcal], dtype=np.float32)
ns.drs_wiggle_shape = np.array([tc.wiggleshape for tc in f.tcal], dtype=np.float32)
ns.drs_mean_shape = np.array([tc.meanshape for tc in f.tcal], dtype=np.float32)

if "hf" in vars():
    hf.close()
    del hf

hf = h5py.File(hdf5_fnam, "w")

def cds(dsetname, comment):
    ds = hf.create_dataset(
        dsetname, 
        data=ns.__dict__[dsetname],
        shuffle=True,
        compression="gzip",
        compression_opts=1)
    ds.attrs["comment"] = comment

cds("drsraw", "raw uncorrected DRS samples [event][channel][sample]")
cds("drs", "DRS samples with pedestal and gain corrections applied")
cds("drsu", "DRS corrected samples, resampled to equal time intervals")
cds("traw", "nominal uncorrected DRS sample times")
cds("tcor", "DRS sample times, corrected via timing calibration")
cds("drs_trig_cell", "DRS trigger/stop cell ID [event][channel]")
cds("drs_tstamp", "CAEN board time stamp [event]")
cds("drs_cellgains", "voltage gain [channel][cell] from DRS calibration")
cds("drs_cellwidth", "cell width (ns) [channel][cell] from DRS timing calibration")
cds("drs_peds", "DRS pedestal [channel][cell] from DRS calibration")
cds("drs_wiggle_shape", "fitted 'wiggle' shape subtracted from each DRS waveform")
cds("drs_mean_shape", "mean waveform, in absence of signal, CAEN board artifact")
hf.close()
del hf
print(f"hdf5 elapsed time: {time.time()-t0:.1f}s")
print(f"total elapsed time {time.time()-starttime:.1f}s")

In [ ]:
starttime = time.time()
nevents = 15000
t0 = time.time()
f.do_triggered_readout(nevents)
print(list(f.trigev[0].__dict__.keys()))
print(f"acquisition elapsed time {time.time()-t0:.1f}s")
print(f"rough rate = {nevents/(time.time()-t0):.1f}Hz")
beep(1)
t0 = time.time()
f.correct_triggered_readout()
print(f"corrections elapsed time {time.time()-t0:.1f}s")
beep(1)

t0 = time.time()
hdf5_fnam = "20250625_nuvmtarray_4x4xtal_47p5V_Ref250mV_Sum400mV_DB1_midleft3x2_15ke_coinc_CAEN_run13.hdf5"

ns = SimpleNamespace()
ns.drsraw = np.array([e.drsraw for e in f.trigev], dtype=np.int16)
ns.drs = np.array([e.drsgc for e in f.trigev], dtype=np.float32)
ns.drsu = np.array([e.drsu for e in f.trigev], dtype=np.float32)
ns.traw = np.array([e.traw for e in f.trigev], dtype=np.float32)
ns.tcor = np.array([e.tcor for e in f.trigev], dtype=np.float32)
ns.drs_trig_cell = np.array([e.drs_trig_cell for e in f.trigev], dtype=np.int16)
ns.drs_tstamp = np.array([e.tstamp for e in f.trigev], dtype=np.int64)
ns.drs_cellgains = np.array([tc.cellgain for tc in f.tcal], dtype=np.float32)
ns.drs_cellwidth = np.array([tc.celldt for tc in f.tcal], dtype=np.float32)
ns.drs_peds = np.array([tc.cellpeds for tc in f.tcal], dtype=np.float32)
ns.drs_wiggle_shape = np.array([tc.wiggleshape for tc in f.tcal], dtype=np.float32)
ns.drs_mean_shape = np.array([tc.meanshape for tc in f.tcal], dtype=np.float32)

if "hf" in vars():
    hf.close()
    del hf

hf = h5py.File(hdf5_fnam, "w")

def cds(dsetname, comment):
    ds = hf.create_dataset(
        dsetname, 
        data=ns.__dict__[dsetname],
        shuffle=True,
        compression="gzip",
        compression_opts=1)
    ds.attrs["comment"] = comment

cds("drsraw", "raw uncorrected DRS samples [event][channel][sample]")
cds("drs", "DRS samples with pedestal and gain corrections applied")
cds("drsu", "DRS corrected samples, resampled to equal time intervals")
cds("traw", "nominal uncorrected DRS sample times")
cds("tcor", "DRS sample times, corrected via timing calibration")
cds("drs_trig_cell", "DRS trigger/stop cell ID [event][channel]")
cds("drs_tstamp", "CAEN board time stamp [event]")
cds("drs_cellgains", "voltage gain [channel][cell] from DRS calibration")
cds("drs_cellwidth", "cell width (ns) [channel][cell] from DRS timing calibration")
cds("drs_peds", "DRS pedestal [channel][cell] from DRS calibration")
cds("drs_wiggle_shape", "fitted 'wiggle' shape subtracted from each DRS waveform")
cds("drs_mean_shape", "mean waveform, in absence of signal, CAEN board artifact")
hf.close()
del hf
print(f"hdf5 elapsed time: {time.time()-t0:.1f}s")
print(f"total elapsed time {time.time()-starttime:.1f}s")

In [ ]:
starttime = time.time()
nevents = 15000
t0 = time.time()
f.do_triggered_readout(nevents)
print(list(f.trigev[0].__dict__.keys()))
print(f"acquisition elapsed time {time.time()-t0:.1f}s")
print(f"rough rate = {nevents/(time.time()-t0):.1f}Hz")
beep(1)
t0 = time.time()
f.correct_triggered_readout()
print(f"corrections elapsed time {time.time()-t0:.1f}s")
beep(1)

t0 = time.time()
hdf5_fnam = "20250625_nuvmtarray_4x4xtal_47p5V_Ref250mV_Sum400mV_DB1_midleft3x2_15ke_coinc_CAEN_run14.hdf5"

ns = SimpleNamespace()
ns.drsraw = np.array([e.drsraw for e in f.trigev], dtype=np.int16)
ns.drs = np.array([e.drsgc for e in f.trigev], dtype=np.float32)
ns.drsu = np.array([e.drsu for e in f.trigev], dtype=np.float32)
ns.traw = np.array([e.traw for e in f.trigev], dtype=np.float32)
ns.tcor = np.array([e.tcor for e in f.trigev], dtype=np.float32)
ns.drs_trig_cell = np.array([e.drs_trig_cell for e in f.trigev], dtype=np.int16)
ns.drs_tstamp = np.array([e.tstamp for e in f.trigev], dtype=np.int64)
ns.drs_cellgains = np.array([tc.cellgain for tc in f.tcal], dtype=np.float32)
ns.drs_cellwidth = np.array([tc.celldt for tc in f.tcal], dtype=np.float32)
ns.drs_peds = np.array([tc.cellpeds for tc in f.tcal], dtype=np.float32)
ns.drs_wiggle_shape = np.array([tc.wiggleshape for tc in f.tcal], dtype=np.float32)
ns.drs_mean_shape = np.array([tc.meanshape for tc in f.tcal], dtype=np.float32)

if "hf" in vars():
    hf.close()
    del hf

hf = h5py.File(hdf5_fnam, "w")

def cds(dsetname, comment):
    ds = hf.create_dataset(
        dsetname, 
        data=ns.__dict__[dsetname],
        shuffle=True,
        compression="gzip",
        compression_opts=1)
    ds.attrs["comment"] = comment

cds("drsraw", "raw uncorrected DRS samples [event][channel][sample]")
cds("drs", "DRS samples with pedestal and gain corrections applied")
cds("drsu", "DRS corrected samples, resampled to equal time intervals")
cds("traw", "nominal uncorrected DRS sample times")
cds("tcor", "DRS sample times, corrected via timing calibration")
cds("drs_trig_cell", "DRS trigger/stop cell ID [event][channel]")
cds("drs_tstamp", "CAEN board time stamp [event]")
cds("drs_cellgains", "voltage gain [channel][cell] from DRS calibration")
cds("drs_cellwidth", "cell width (ns) [channel][cell] from DRS timing calibration")
cds("drs_peds", "DRS pedestal [channel][cell] from DRS calibration")
cds("drs_wiggle_shape", "fitted 'wiggle' shape subtracted from each DRS waveform")
cds("drs_mean_shape", "mean waveform, in absence of signal, CAEN board artifact")
hf.close()
del hf
print(f"hdf5 elapsed time: {time.time()-t0:.1f}s")
print(f"total elapsed time {time.time()-starttime:.1f}s")

In [ ]:
starttime = time.time()
nevents = 15000
t0 = time.time()
f.do_triggered_readout(nevents)
print(list(f.trigev[0].__dict__.keys()))
print(f"acquisition elapsed time {time.time()-t0:.1f}s")
print(f"rough rate = {nevents/(time.time()-t0):.1f}Hz")
beep(1)
t0 = time.time()
f.correct_triggered_readout()
print(f"corrections elapsed time {time.time()-t0:.1f}s")
beep(1)

t0 = time.time()
hdf5_fnam = "20250625_nuvmtarray_4x4xtal_47p5V_Ref250mV_Sum400mV_DB1_midleft3x2_15ke_coinc_CAEN_run15.hdf5"

ns = SimpleNamespace()
ns.drsraw = np.array([e.drsraw for e in f.trigev], dtype=np.int16)
ns.drs = np.array([e.drsgc for e in f.trigev], dtype=np.float32)
ns.drsu = np.array([e.drsu for e in f.trigev], dtype=np.float32)
ns.traw = np.array([e.traw for e in f.trigev], dtype=np.float32)
ns.tcor = np.array([e.tcor for e in f.trigev], dtype=np.float32)
ns.drs_trig_cell = np.array([e.drs_trig_cell for e in f.trigev], dtype=np.int16)
ns.drs_tstamp = np.array([e.tstamp for e in f.trigev], dtype=np.int64)
ns.drs_cellgains = np.array([tc.cellgain for tc in f.tcal], dtype=np.float32)
ns.drs_cellwidth = np.array([tc.celldt for tc in f.tcal], dtype=np.float32)
ns.drs_peds = np.array([tc.cellpeds for tc in f.tcal], dtype=np.float32)
ns.drs_wiggle_shape = np.array([tc.wiggleshape for tc in f.tcal], dtype=np.float32)
ns.drs_mean_shape = np.array([tc.meanshape for tc in f.tcal], dtype=np.float32)

if "hf" in vars():
    hf.close()
    del hf

hf = h5py.File(hdf5_fnam, "w")

def cds(dsetname, comment):
    ds = hf.create_dataset(
        dsetname, 
        data=ns.__dict__[dsetname],
        shuffle=True,
        compression="gzip",
        compression_opts=1)
    ds.attrs["comment"] = comment

cds("drsraw", "raw uncorrected DRS samples [event][channel][sample]")
cds("drs", "DRS samples with pedestal and gain corrections applied")
cds("drsu", "DRS corrected samples, resampled to equal time intervals")
cds("traw", "nominal uncorrected DRS sample times")
cds("tcor", "DRS sample times, corrected via timing calibration")
cds("drs_trig_cell", "DRS trigger/stop cell ID [event][channel]")
cds("drs_tstamp", "CAEN board time stamp [event]")
cds("drs_cellgains", "voltage gain [channel][cell] from DRS calibration")
cds("drs_cellwidth", "cell width (ns) [channel][cell] from DRS timing calibration")
cds("drs_peds", "DRS pedestal [channel][cell] from DRS calibration")
cds("drs_wiggle_shape", "fitted 'wiggle' shape subtracted from each DRS waveform")
cds("drs_mean_shape", "mean waveform, in absence of signal, CAEN board artifact")
hf.close()
del hf
print(f"hdf5 elapsed time: {time.time()-t0:.1f}s")
print(f"total elapsed time {time.time()-starttime:.1f}s")

In [ ]:
starttime = time.time()
nevents = 15000
t0 = time.time()
f.do_triggered_readout(nevents)
print(list(f.trigev[0].__dict__.keys()))
print(f"acquisition elapsed time {time.time()-t0:.1f}s")
print(f"rough rate = {nevents/(time.time()-t0):.1f}Hz")
beep(1)
t0 = time.time()
f.correct_triggered_readout()
print(f"corrections elapsed time {time.time()-t0:.1f}s")
beep(1)

t0 = time.time()
hdf5_fnam = "20250625_nuvmtarray_4x4xtal_47p5V_Ref250mV_Sum400mV_DB1_midleft3x2_15ke_coinc_CAEN_run16.hdf5"

ns = SimpleNamespace()
ns.drsraw = np.array([e.drsraw for e in f.trigev], dtype=np.int16)
ns.drs = np.array([e.drsgc for e in f.trigev], dtype=np.float32)
ns.drsu = np.array([e.drsu for e in f.trigev], dtype=np.float32)
ns.traw = np.array([e.traw for e in f.trigev], dtype=np.float32)
ns.tcor = np.array([e.tcor for e in f.trigev], dtype=np.float32)
ns.drs_trig_cell = np.array([e.drs_trig_cell for e in f.trigev], dtype=np.int16)
ns.drs_tstamp = np.array([e.tstamp for e in f.trigev], dtype=np.int64)
ns.drs_cellgains = np.array([tc.cellgain for tc in f.tcal], dtype=np.float32)
ns.drs_cellwidth = np.array([tc.celldt for tc in f.tcal], dtype=np.float32)
ns.drs_peds = np.array([tc.cellpeds for tc in f.tcal], dtype=np.float32)
ns.drs_wiggle_shape = np.array([tc.wiggleshape for tc in f.tcal], dtype=np.float32)
ns.drs_mean_shape = np.array([tc.meanshape for tc in f.tcal], dtype=np.float32)

if "hf" in vars():
    hf.close()
    del hf

hf = h5py.File(hdf5_fnam, "w")

def cds(dsetname, comment):
    ds = hf.create_dataset(
        dsetname, 
        data=ns.__dict__[dsetname],
        shuffle=True,
        compression="gzip",
        compression_opts=1)
    ds.attrs["comment"] = comment

cds("drsraw", "raw uncorrected DRS samples [event][channel][sample]")
cds("drs", "DRS samples with pedestal and gain corrections applied")
cds("drsu", "DRS corrected samples, resampled to equal time intervals")
cds("traw", "nominal uncorrected DRS sample times")
cds("tcor", "DRS sample times, corrected via timing calibration")
cds("drs_trig_cell", "DRS trigger/stop cell ID [event][channel]")
cds("drs_tstamp", "CAEN board time stamp [event]")
cds("drs_cellgains", "voltage gain [channel][cell] from DRS calibration")
cds("drs_cellwidth", "cell width (ns) [channel][cell] from DRS timing calibration")
cds("drs_peds", "DRS pedestal [channel][cell] from DRS calibration")
cds("drs_wiggle_shape", "fitted 'wiggle' shape subtracted from each DRS waveform")
cds("drs_mean_shape", "mean waveform, in absence of signal, CAEN board artifact")
hf.close()
del hf
print(f"hdf5 elapsed time: {time.time()-t0:.1f}s")
print(f"total elapsed time {time.time()-starttime:.1f}s")

In [ ]:
starttime = time.time()
nevents = 15000
t0 = time.time()
f.do_triggered_readout(nevents)
print(list(f.trigev[0].__dict__.keys()))
print(f"acquisition elapsed time {time.time()-t0:.1f}s")
print(f"rough rate = {nevents/(time.time()-t0):.1f}Hz")
beep(1)
t0 = time.time()
f.correct_triggered_readout()
print(f"corrections elapsed time {time.time()-t0:.1f}s")
beep(1)

t0 = time.time()
hdf5_fnam = "20250625_nuvmtarray_4x4xtal_47p5V_Ref250mV_Sum400mV_DB1_midleft3x2_15ke_coinc_CAEN_run17.hdf5"

ns = SimpleNamespace()
ns.drsraw = np.array([e.drsraw for e in f.trigev], dtype=np.int16)
ns.drs = np.array([e.drsgc for e in f.trigev], dtype=np.float32)
ns.drsu = np.array([e.drsu for e in f.trigev], dtype=np.float32)
ns.traw = np.array([e.traw for e in f.trigev], dtype=np.float32)
ns.tcor = np.array([e.tcor for e in f.trigev], dtype=np.float32)
ns.drs_trig_cell = np.array([e.drs_trig_cell for e in f.trigev], dtype=np.int16)
ns.drs_tstamp = np.array([e.tstamp for e in f.trigev], dtype=np.int64)
ns.drs_cellgains = np.array([tc.cellgain for tc in f.tcal], dtype=np.float32)
ns.drs_cellwidth = np.array([tc.celldt for tc in f.tcal], dtype=np.float32)
ns.drs_peds = np.array([tc.cellpeds for tc in f.tcal], dtype=np.float32)
ns.drs_wiggle_shape = np.array([tc.wiggleshape for tc in f.tcal], dtype=np.float32)
ns.drs_mean_shape = np.array([tc.meanshape for tc in f.tcal], dtype=np.float32)

if "hf" in vars():
    hf.close()
    del hf

hf = h5py.File(hdf5_fnam, "w")

def cds(dsetname, comment):
    ds = hf.create_dataset(
        dsetname, 
        data=ns.__dict__[dsetname],
        shuffle=True,
        compression="gzip",
        compression_opts=1)
    ds.attrs["comment"] = comment

cds("drsraw", "raw uncorrected DRS samples [event][channel][sample]")
cds("drs", "DRS samples with pedestal and gain corrections applied")
cds("drsu", "DRS corrected samples, resampled to equal time intervals")
cds("traw", "nominal uncorrected DRS sample times")
cds("tcor", "DRS sample times, corrected via timing calibration")
cds("drs_trig_cell", "DRS trigger/stop cell ID [event][channel]")
cds("drs_tstamp", "CAEN board time stamp [event]")
cds("drs_cellgains", "voltage gain [channel][cell] from DRS calibration")
cds("drs_cellwidth", "cell width (ns) [channel][cell] from DRS timing calibration")
cds("drs_peds", "DRS pedestal [channel][cell] from DRS calibration")
cds("drs_wiggle_shape", "fitted 'wiggle' shape subtracted from each DRS waveform")
cds("drs_mean_shape", "mean waveform, in absence of signal, CAEN board artifact")
hf.close()
del hf
print(f"hdf5 elapsed time: {time.time()-t0:.1f}s")
print(f"total elapsed time {time.time()-starttime:.1f}s")

In [ ]:
beep(2)

In [ ]:
import subprocess
from datetime import datetime

notebook = "sipm_array_daq.ipynb"
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
output = f"sipm_array_daq_{timestamp}.html"

subprocess.run(
    ["jupyter", "nbconvert", "--to", "html", notebook, "--output", output],
    check=True
)
print(f"Exported: {output}")

In [ ]:
run export_to_html --timestamp